# Original unknown-$R_p$ workflow: hybrid SciPy and Pyomo fitting

This tutorial starts from the original measured vial-bottom temperature series. Both backends share LyoPRONTO's legacy inverse-temperature preprocessing, which derives cake length $L_{ck}$ [cm] and product resistance $R_p$ [cm$^2$ hr Torr/g]. SciPy `curve_fit` and Pyomo then fit the same observations to

$$R_p = R_0 + \frac{A_1 L_{ck}}{1 + A_2 L_{ck}}.$$

This is explicitly a **hybrid** Pyomo workflow: Pyomo replaces only the nonlinear parameter fit. A direct Pyomo inverse model for measured $T_{bot}(t)$ is outside the current model boundary.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from examples.original_workflow_parity import (
    fit_unknown_rp_pyomo,
    fit_unknown_rp_scipy,
    preprocess_unknown_rp,
    pyomo_ipopt_status,
)

In [ ]:
trajectory, product_resistance = preprocess_unknown_rp()
scipy_fit = fit_unknown_rp_scipy(product_resistance)
print(f"Legacy preprocessing: {len(product_resistance)} observations")
print(f"SciPy R0/A1/A2: {scipy_fit.R0:.8f}, {scipy_fit.A1:.8f}, {scipy_fit.A2:.8f}")
print(f"SciPy sum of squared Rp residuals: {scipy_fit.objective:.8f}")

In [ ]:
pyomo_ready, pyomo_message = pyomo_ipopt_status()
print(pyomo_message)
pyomo_fit = fit_unknown_rp_pyomo(product_resistance) if pyomo_ready else None
if pyomo_fit is not None:
    print(f"Pyomo R0/A1/A2: {pyomo_fit.R0:.8f}, {pyomo_fit.A1:.8f}, {pyomo_fit.A2:.8f}")
    print(f"Pyomo status: {pyomo_fit.solver_status}/{pyomo_fit.termination_condition}")
    print(f"Pyomo sum of squared Rp residuals: {pyomo_fit.objective:.8f}")

The solver-backed regression allows $2\times10^{-5}$ relative or absolute parameter error and $10^{-6}$ absolute objective error. These tolerances cover local NLP termination while remaining much tighter than the scale of the fitted parameters.

In [ ]:
if pyomo_fit is not None:
    np.testing.assert_allclose(pyomo_fit.as_array(), scipy_fit.as_array(), rtol=2e-5, atol=2e-5)
    assert abs(pyomo_fit.objective - scipy_fit.objective) <= 1e-6
    print("SciPy/Pyomo parameter and objective tolerances satisfied.")

In [ ]:
length = product_resistance[:, 1]
observed_rp = product_resistance[:, 2]
scipy_curve = scipy_fit.R0 + length * scipy_fit.A1 / (1 + length * scipy_fit.A2)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(length, observed_rp, ".", alpha=0.4, label="legacy preprocessing")
axes[0].plot(length, scipy_curve, label="SciPy fit")
if pyomo_fit is not None:
    pyomo_curve = pyomo_fit.R0 + length * pyomo_fit.A1 / (1 + length * pyomo_fit.A2)
    axes[0].plot(length, pyomo_curve, "--", label="Pyomo fit")
axes[1].plot(trajectory[:, 0], trajectory[:, 2], label="measured-replay $T_{bot}$")
axes[0].set(xlabel="Cake length [cm]", ylabel="$R_p$ [cm$^2$ hr Torr/g]")
axes[1].set(xlabel="Time [hr]", ylabel="Temperature [degC]")
for axis in axes:
    axis.grid(alpha=0.3)
    axis.legend()
fig.tight_layout()